In [1]:
!pip install -q transformers trl peft accelerate datasets huggingface_hub wandb

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: /Users/jagath-sajjan/BugginHell


In [3]:
import re
import json
import torch
from datasets import Dataset

from bughunt_env import BugHuntEnv

In [4]:
# Promptt buidlrrrr

def build_prompt(obs):
    return f"""
You are BugHunt RL, an agent debugging a Python repo.

You can output exactly ONE action in JSON.

Allowed actions:
1. {{"tool": "read_file", "path": "file.py"}}
2. {{"tool": "run_test", "name": "test_name"}}
3. {{"tool": "search_symbol", "name": "symbol_name"}}
4. {{"tool": "trace_caller", "fn": "function_name"}}
5. {{"tool": "commit_location", "file": "file.py", "line": 123}}

Repo files:
{obs.file_tree}

Failing test:
{obs.failing_test}

Test error:
{obs.stderr}

Last tool output:
{obs.last_tool_output}

Steps left:
{obs.steps_left}

Return only JSON.
""".strip()

In [5]:
# Parse Model Actionnn

ACTION_MAP = {
    "read_file": 0,
    "run_test": 1,
    "search_symbol": 2,
    "trace_caller": 3,
    "commit_location": 4,
}

def parse_action(text, obs):
    try:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if not match:
            raise ValueError("No JSON found")

        data = json.loads(match.group(0))
        tool = data.get("tool")

        if tool == "read_file":
            return (0, {"path": data.get("path", obs.file_tree[0])})

        if tool == "run_test":
            return (1, {"name": data.get("name", obs.failing_test)})

        if tool == "search_symbol":
            return (2, {"name": data.get("name", obs.failing_test)})

        if tool == "trace_caller":
            return (3, {"fn": data.get("fn", "")})

        if tool == "commit_location":
            return (
                4,
                {
                    "file": data.get("file", ""),
                    "line": int(data.get("line", -999)),
                },
            )

    except Exception:
        pass

    return (1, {"name": obs.failing_test})

In [6]:
# Rollout — full episode, not a single step

def evaluate_completion(completion, seed=1):
    """
    Run the model's completion through a FULL episode.
    The completion is the first action; subsequent actions reuse the same
    greedy parse so the agent plays out the rest of the episode.
    This gives GRPO a reward signal that reflects multi-step consequences,
    not just the quality of one isolated move.
    """
    env = BugHuntEnv(seed=seed)
    obs, info = env.reset(seed=seed)

    total_reward = 0.0

    # First action comes from the model's completion
    action = parse_action(completion, obs)
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward

    # Continue the episode with greedy re-parses of the same prompt logic
    # (simulates the agent continuing after its initial move)
    while not (terminated or truncated):
        # Re-prompt with updated obs to get next action
        next_prompt = build_prompt(obs)
        # Re-use the same parse — in real GRPO the model would be called again,
        # but for reward scoring we do a lightweight re-parse from obs heuristic
        import re as _re, json as _json
        # Derive a sensible fallback action from the current observation
        if obs.steps_left <= 1:
            # Out of budget — force a commit based on best info so far
            tool_out = obs.last_tool_output
            file_m = _re.search(r'found in ([^\s]+\.py)', tool_out.lower())
            src_files = [f for f in obs.file_tree if f.endswith('.py') and 'test' not in f]
            best_file = file_m.group(1) if file_m else (src_files[0] if src_files else obs.file_tree[0])
            fallback = (4, {'file': best_file, 'line': 1})
        elif not obs.history:
            fallback = (2, {'name': obs.failing_test})  # search symbol first
        elif len(obs.history) == 1:
            sym_m = _re.search(r'found in ([^\s]+\.py)', obs.last_tool_output.lower())
            fn_matches = _re.findall(r'\b([a-z_][a-z0-9_]{3,})\s*[\(=]', obs.stderr.lower())
            noise = {'assert', 'where', 'from', 'import', 'true', 'false', 'none'}
            fn = next((s for s in fn_matches if s not in noise), obs.failing_test)
            fallback = (3, {'fn': fn})  # trace caller
        else:
            tool_out = obs.last_tool_output.lower()
            file_m = _re.search(r'from ([^\s]+\.py)', tool_out) or _re.search(r'found in ([^\s]+\.py)', tool_out)
            src_files = [f for f in obs.file_tree if f.endswith('.py') and 'test' not in f]
            best_file = file_m.group(1) if file_m else (src_files[0] if src_files else obs.file_tree[0])
            fallback = (0, {'path': best_file})  # read the file

        obs, reward, terminated, truncated, info = env.step(fallback)
        total_reward += reward

    return float(total_reward)


In [7]:
# Dataset of env states — includes mid-episode observations, not just step-0

def make_dataset(n=100):
    """
    Build training prompts from diverse environment states.
    For each seed we collect:
      - The initial observation (step 0)
      - Up to 2 mid-episode observations (after search_symbol and trace_caller)
    This teaches the model what to do at every stage of debugging,
    not just the very first move.
    """
    prompts = []

    for i in range(n):
        env = BugHuntEnv(seed=i)
        obs, info = env.reset(seed=i)

        # Step 0: initial state
        prompts.append({"prompt": build_prompt(obs), "seed": i})

        # Step 1: after search_symbol
        import re as _re
        fn_matches = _re.findall(r'\b([a-z_][a-z0-9_]{3,})\s*[\(=]', obs.stderr.lower())
        noise = {'assert', 'where', 'from', 'import', 'true', 'false', 'none'}
        fn = next((s for s in fn_matches if s not in noise), obs.failing_test)
        obs, _, terminated, truncated, _ = env.step((2, {'name': fn}))

        if not (terminated or truncated):
            prompts.append({"prompt": build_prompt(obs), "seed": i})

            # Step 2: after trace_caller
            obs, _, terminated, truncated, _ = env.step((3, {'fn': fn}))
            if not (terminated or truncated):
                prompts.append({"prompt": build_prompt(obs), "seed": i})

    return Dataset.from_list(prompts)

train_dataset = make_dataset(100)
print(f"Dataset size: {len(train_dataset)} prompts (was 100 before, now includes mid-episode states)")
train_dataset[0]


{'prompt': 'You are BugHunt RL, an agent debugging a Python repo.\n\nYou can output exactly ONE action in JSON.\n\nAllowed actions:\n1. {"tool": "read_file", "path": "file.py"}\n2. {"tool": "run_test", "name": "test_name"}\n3. {"tool": "search_symbol", "name": "symbol_name"}\n4. {"tool": "trace_caller", "fn": "function_name"}\n5. {"tool": "commit_location", "file": "file.py", "line": 123}\n\nRepo files:\n[\'README.md\', \'auth.py\', \'tests/test_auth.py\', \'profile.py\']\n\nFailing test:\ntest_admin_user_is_admin\n\nTest error:\nFAILED tests/test_auth.py::test_admin_user_is_admin\\nE assert False is True\\nE where False = is_admin({\'role\': \'admin\'})\n\nLast tool output:\n\n\nSteps left:\n10\n\nReturn only JSON.',
 'seed': 0}

In [8]:
# GRPO Reward Func

def bughunt_reward_func(prompts, completions, seed=None, **kwargs):
    rewards = []

    seeds = kwargs.get("seed", [1] * len(completions))

    for completion, s in zip(completions, seeds):
        if isinstance(completion, list):
            text = completion[0].get("content", "")
        else:
            text = str(completion)

        reward = evaluate_completion(text, seed=int(s))
        rewards.append(reward)

    return rewards

In [1]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

if device != "cuda":
    raise RuntimeError("Stop here. GRPO training needs Colab/HuggingFace GPU. Do not run trainer.train() locally.")

Device: cpu


RuntimeError: Stop here. GRPO training needs Colab/HuggingFace GPU. Do not run trainer.train() locally.

In [9]:
# GRPO Trainer Setup

from trl import GRPOTrainer, GRPOConfig

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

training_args = GRPOConfig(
    output_dir=str(PROJECT_ROOT / "outputs" / "grpo-bugginhell"),
    learning_rate=5e-6,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_generations=2,
    max_prompt_length=1024,
    max_completion_length=128,
    num_train_epochs=1,
    logging_steps=1,
    save_steps=25,
    report_to="none",
)

trainer = GRPOTrainer(
    model=MODEL_NAME,
    reward_funcs=bughunt_reward_func,
    args=training_args,
    train_dataset=train_dataset,
)

W0426 01:33:40.895000 73755 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [10]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/Users/jagath-sajjan/.pyenv/versions/3.12.7/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


RuntimeError: MPS backend out of memory (MPS allocated: 8.35 GiB, other allocations: 549.61 MiB, max allowed: 9.07 GiB). Tried to allocate 519.31 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

In [ ]:
trainer.save_model(str(PROJECT_ROOT / "outputs" / "bugginhell-grpo-agent"))